# deep_taste — restaurant feature encode (GPU)

Runs `features.py` on a Colab GPU instead of local MPS. Encodes ~397k reviews with
`thenlper/gte-base` and pools them with exponential recency weights.

**Before running — set the runtime to a GPU:** Runtime → Change runtime type → T4 GPU.

**Upload these 3 files to Google Drive in a folder named `deep_taste`:**
- `data/processed/reviews_split.parquet` (179 MB)
- `data/processed/businesses.parquet` (464 KB)
- `src/features.py` (8 KB)

In [ ]:
# 1. Confirm we actually got a GPU. If this errors, the runtime is still CPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Deps (torch is preinstalled on Colab)
!pip install -q sentence-transformers

In [ ]:
# 3. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Stage inputs onto local disk (Drive I/O is slow to read from directly)
DRIVE = '/content/drive/MyDrive/deep_taste'
!mkdir -p /content/data
!cp $DRIVE/reviews_split.parquet $DRIVE/businesses.parquet /content/data/
!cp $DRIVE/features.py /content/
!ls -la /content/data/

In [ ]:
# 5. The encode. ~15-25 min on a T4.
# fp16 roughly doubles throughput on CUDA; batch 128 fits T4 memory at 512 tokens.
!python -u /content/features.py --data-dir /content/data --tau 2.0 --batch-size 128 --fp16

In [ ]:
# 6. Sanity-check before spending bandwidth on the copy back
import torch
d = torch.load('/content/data/features.pt', weights_only=False)
for k, v in d.items():
    if torch.is_tensor(v):
        print(f'{k:12} {str(tuple(v.shape)):16} {v.dtype}')
assert d['text_emb'].shape[1] == 768, 'wrong embedding dim'
assert not torch.isnan(d['text_emb']).any(), 'NaNs in text_emb'
print('\nOK')

In [ ]:
# 7. Copy results back to Drive.
# features.pt (~20 MB) is what training needs.
# review_vecs.pt (~610 MB) is only for the learned-attention upgrade later --
# leave it in Drive rather than downloading it now.
!cp /content/data/features.pt /content/data/norm_stats.json $DRIVE/
!cp /content/data/review_vecs.pt $DRIVE/
!ls -lah $DRIVE/